# Implementasi Sistem Fuzzy - Presentasi
Notebook ini berisi potongan kode implementasi untuk presentasi

In [ ]:
import pandas as pd
import numpy as np

# Contoh data untuk testing
gaji = 3.5  # 3.5 juta
tanggungan = 50  # 50% cicilan
g, t = gaji, tanggungan

## 1. Fungsi Membership Dasar

In [ ]:
# Fungsi keanggotaan trapesium
def trapezoid(x, a, b, c, d):
    if x <= a or x >= d:
        return 0
    elif a < x <= b:
        return (x - a) / (b - a)
    elif b < x < c:
        return 1
    elif c <= x < d:
        return (d - x) / (d - c)

# Fungsi keanggotaan segitiga
def triangle(x, a, b, c):
    if x <= a or x >= c:
        return 0
    elif a < x <= b:
        return (x - a) / (b - a)
    elif b < x < c:
        return (c - x) / (c - b)

# Metode Mamdani - Resiko Kredit

In [ ]:
# Fuzzifikasi Input
gaji_rendah = trapezoid(gaji, 0, 0, 2, 4)
gaji_sedang = triangle(gaji, 2, 5, 8)
gaji_tinggi = trapezoid(gaji, 6, 8, 10, 10)

tanggungan_sedikit = trapezoid(tanggungan, 0, 0, 40, 48)
tanggungan_sedang = triangle(tanggungan, 45, 50, 55)
tanggungan_banyak = trapezoid(tanggungan, 52, 58, 100, 100)

# Aturan Fuzzy (9 rules)
aturan1 = min(gaji_rendah, tanggungan_banyak)   # Resiko Tinggi
aturan2 = min(gaji_rendah, tanggungan_sedang)   # Resiko Tinggi
aturan3 = min(gaji_rendah, tanggungan_sedikit)  # Resiko Sedang
aturan4 = min(gaji_sedang, tanggungan_banyak)   # Resiko Sedang
aturan5 = min(gaji_sedang, tanggungan_sedang)   # Resiko Sedang
aturan6 = min(gaji_sedang, tanggungan_sedikit)  # Resiko Rendah
aturan7 = min(gaji_tinggi, tanggungan_banyak)   # Resiko Rendah
aturan8 = min(gaji_tinggi, tanggungan_sedang)   # Resiko Rendah
aturan9 = min(gaji_tinggi, tanggungan_sedikit)  # Resiko Rendah

# Agregasi Output
resiko_tinggi = max(aturan1, aturan2)
resiko_sedang = max(aturan3, aturan4, aturan5)
resiko_rendah = max(aturan6, aturan7, aturan8, aturan9)

# Defuzzifikasi (Centroid Method)
x_tinggi = 80
x_sedang = 50
x_rendah = 20

numerator = (resiko_tinggi * x_tinggi) + (resiko_sedang * x_sedang) + (resiko_rendah * x_rendah)
denominator = resiko_tinggi + resiko_sedang + resiko_rendah
resiko_crisp = numerator / denominator

# Metode Tsukamoto - Beasiswa

# Metode Mamdani - Beasiswa

In [ ]:
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# Definisi variabel fuzzy
ipk = ctrl.Antecedent(np.arange(0, 4.1, 0.1), 'ipk')
toefl = ctrl.Antecedent(np.arange(250, 601, 1), 'toefl')
income = ctrl.Antecedent(np.arange(0, 2.1, 0.01), 'income')
eligibility = ctrl.Consequent(np.arange(0, 101, 1), 'eligibility')

# Fungsi keanggotaan
ipk['cukup'] = fuzz.trimf(ipk.universe, [2.5, 3.0, 3.5])
ipk['bagus'] = fuzz.trimf(ipk.universe, [3.0, 3.5, 4.0])
toefl['menengah'] = fuzz.trimf(toefl.universe, [400, 450, 500])
toefl['tinggi'] = fuzz.trimf(toefl.universe, [500, 550, 600])
income['kecil'] = fuzz.trapmf(income.universe, [0, 0, 0.6, 1.0])
income['sedang'] = fuzz.trimf(income.universe, [0.8, 1.2, 1.6])
income['besar'] = fuzz.trimf(income.universe, [1.4, 1.7, 2.0])
income['sangat_besar'] = fuzz.trapmf(income.universe, [1.8, 2.0, 2.0, 2.0])
eligibility['rendah'] = fuzz.trimf(eligibility.universe, [0, 25, 50])
eligibility['tinggi'] = fuzz.trimf(eligibility.universe, [50, 75, 100])

# Rules lengkap (5 rules)
rule1 = ctrl.Rule(ipk['bagus'] & (income['kecil'] | income['sedang']), eligibility['tinggi'])
rule2 = ctrl.Rule(ipk['bagus'] & (income['besar'] | income['sangat_besar']), eligibility['rendah'])
rule3 = ctrl.Rule(ipk['cukup'] & (income['kecil'] | income['sedang']), eligibility['rendah'])
rule4 = ctrl.Rule(toefl['tinggi'] & (income['kecil'] | income['sedang']), eligibility['tinggi'])
rule5 = ctrl.Rule(toefl['menengah'] & (income['kecil'] | income['sedang']), eligibility['tinggi'])

# Sistem kontrol
system = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5])
sim = ctrl.ControlSystemSimulation(system)

# Testing
sim.input['ipk'] = 4.0
sim.input['toefl'] = 450
sim.input['income'] = 0.75
sim.compute()
print(f"Kelayakan Beasiswa: {sim.output['eligibility']:.2f}")

In [ ]:
# Fungsi membership MONOTON
def membership_naik(alpha, a, b):
    # Untuk output tinggi (monoton naik)
    return a + alpha * (b - a)

def membership_turun(alpha, a, b):
    # Untuk output rendah (monoton turun)
    return b - alpha * (b - a)

In [ ]:
# Fuzzifikasi manual IPK
if ipk_val <= 3.0:
    mu_ipk_bagus = 0
elif 3.0 < ipk_val <= 3.5:
    mu_ipk_bagus = (ipk_val - 3.0) / (3.5 - 3.0)
elif 3.5 < ipk_val <= 4.0:
    mu_ipk_bagus = (4.0 - ipk_val) / (4.0 - 3.5)

# Evaluasi rule Tsukamoto
alpha1 = min(mu_ipk_bagus, max(mu_income_kecil, mu_income_sedang))
if alpha1 > 0:
    z1 = membership_naik(alpha1, 50, 100)  # inverse membership
    rules_z.append(z1)
    rules_alpha.append(alpha1)

# Defuzzifikasi Weighted Average
z_akhir = sum([z * a for z, a in zip(rules_z, rules_alpha)]) / sum(rules_alpha)